## Аудит метаданных

In [1]:
# импорты
import pandas as pd
from src.config import DOCS
from src.data_loader import load_parquet

c:\Users\Dias\Desktop\final_programming\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# загружаем весь датасет
df = load_parquet()
df.shape

(215889, 24)

In [3]:
#пропуски и заглушки
PLACEHOLDERS = ["ЖОҚ", "НЕТ", "нет", "undefined"]

# список, куда складываем по одной строке-словарю на каждую колонку
rows = []

for col in df.columns:
    nan_count = df[col].isna().sum()

    placeholder_count = df[col].isin(PLACEHOLDERS).sum()

# Пропуски заглушки статистикой в словарь(потом в таблицу)
    rows.append({
        "column": col,
        "nan": nan_count,
        "placeholder": placeholder_count,
        "missing_total": nan_count + placeholder_count,
    })

missing = pd.DataFrame(rows) 

missing["missing_pct"] = (missing["missing_total"] / len(df) * 100).round(1)

missing = missing.sort_values("missing_pct", ascending=False)
missing

,column,nan,placeholder,missing_total,missing_pct
4,date_published,190769,0,190769,88.4
8,ministry_registration_date,186994,0,186994,86.6
20,developer_body,148355,3821,152176,70.5
3,date_modified,106980,0,106980,49.6
7,gov_registration_number,25126,78774,103900,48.1
19,regulatory_body,19,93272,93291,43.2
14,official_publication_info,951,40656,41607,19.3
11,ekb_url,33742,0,33742,15.6
5,state_registry_number,30868,0,30868,14.3
10,adilet_url,794,0,794,0.4


In [4]:
# сохраняем отчёт о пропусках
missing.to_csv(DOCS / "audit_missing.csv", index=False, encoding="utf-8-sig")

**Результат (пропуски):**
- Больше всего пропусков в `date_published` (88.4 %), `ministry_registration_date` (86.6 %), `developer_body` (70.5 %).
- В `regulatory_body` всего 19 пустых ячеек, но 93 272 заглушки `ЖОҚ`, то есть на деле пусто 43.2 %. Если считать только NaN, этого не видно.
- `date_adopted`, `title`, `act_form`, `adopting_body`, `region` заполнены почти полностью — по ним можно строить анализ.

In [5]:
# диапазон дат
DATE_COLS = ["date_adopted", "date_modified", "date_published", "ministry_registration_date"]
df[DATE_COLS].agg(["min", "max"]).T

,min,max
date_adopted,1947-10-30,2026-07-20
date_modified,1992-09-30,2026-03-10
date_published,2013-12-26,2016-12-30
ministry_registration_date,1997-04-21,2026-07-20


In [6]:
# число актов по годам
years = df["date_adopted"].dt.year.value_counts().sort_index()
print(years.loc[1985:1995], "\n")
print(years.tail(4))

date_adopted
1985       5
1986       9
1987       7
1988       6
1989       6
1990       9
1991      18
1992     241
1993     684
1994     705
1995    1055
Name: count, dtype: int64 

date_adopted
2023    15092
2024    14279
2025    13715
2026     1563
Name: count, dtype: int64


In [7]:
# противоречивые даты
print("изменён раньше принятия:", (df["date_modified"] < df["date_adopted"]).sum())
print("опубликован раньше принятия:", (df["date_published"] < df["date_adopted"]).sum())

изменён раньше принятия: 2
опубликован раньше принятия: 1


**Результат (даты):**
- Все даты в формате `ГГГГ-ММ-ДД`, `date_adopted` — с 1947 по 2026.
- До 1992 года актов единицы (до независимости база почти пустая), 2026 год неполный (1 563 акта против ~14 тыс.). Для трендов и прогноза берём **1992–2025**.
- `date_published` есть только за 2013–2016 — для анализа не подходит.
- Противоречивых дат всего 3 — на результат не влияют.

In [8]:
# правило деления: запятая + заглавная
SEP = r",\s+(?=[A-ZА-ЯЁӘҒҚҢӨҰҮҺІ])"
import re
re.split(SEP, "Су көлігі, Көліктегі қауіпсіздік, Нормативтік құқықтық актілер: дайындау, қабылдау")

['Су көлігі',
 'Көліктегі қауіпсіздік',
 'Нормативтік құқықтық актілер: дайындау, қабылдау']

In [9]:
# статистика неатомарного поля
def multivalue_stats(series, name):
    """Сколько значений в ячейке и сколько уникальных после деления."""
    # не пусто И не заглушка
    s = series.astype(str).where(series.notna() & ~series.isin(PLACEHOLDERS))
    parts = s.dropna().str.split(SEP, regex=True)
    counts = parts.str.len()
    atomic = parts.explode().str.strip()
    return {
        "field": name,
        "rows": len(parts),
        "multi_pct": round((counts > 1).mean() * 100, 1),
        "max_in_cell": counts.max(),
        "unique_raw": s.nunique(),
        "unique_atomic": atomic.nunique(),
    }

In [10]:
# органы без скобок
bodies = df["adopting_body"].astype(str).str.replace(r"\s*\([^()]*\)", "", regex=True)
bodies = bodies.where(df["adopting_body"].notna())

In [11]:
# таблица по неатомарным полям
multi = pd.DataFrame([
    multivalue_stats(df["legal_relation_area"], "legal_relation_area"),
    multivalue_stats(df["act_form"], "act_form"),
    multivalue_stats(bodies, "adopting_body"),
])

# сохраняем отчёт
multi.to_csv(DOCS / "audit_multivalue.csv", index=False, encoding="utf-8-sig")
multi

,field,rows,multi_pct,max_in_cell,unique_raw,unique_atomic
0,legal_relation_area,215509,70.8,12,25283,351
1,act_form,215826,42.6,7,861,75
2,adopting_body,215819,1.7,13,4049,3235


**Результат (неатомарные поля):**
- `legal_relation_area`: в 71 % строк несколько сфер (до 12 в одной ячейке). 25 283 «разных» значения — это комбинации, настоящих сфер всего 351.
- `act_form`: 43 % составные, 861 комбинация → 75 настоящих форм. `Өзгерту` (изменение) — не форма, а пометка о поправках.
- `adopting_body`: после удаления скобок 4 049 → 3 235 органов; несколько органов в одной ячейке только у 1.7 % строк.
- Все три поля нарушают 1NF → в базе выносим их в отдельные таблицы со связью «многие-ко-многим».

In [12]:
# проверяем функции очистки
from src.cleaning import clean_placeholders, strip_brackets, split_values

clean = clean_placeholders(df)
clean["adopting_body"] = strip_brackets(clean["adopting_body"])

print("regulatory_body пропусков:", clean["regulatory_body"].isna().sum())
for col in ["legal_relation_area", "act_form", "adopting_body"]:
    pairs = split_values(clean, col)
    print(col, "пар:", len(pairs), "уникальных:", pairs["name"].nunique())

regulatory_body пропусков: 93291
legal_relation_area пар: 428099 уникальных: 351
act_form пар: 320143 уникальных: 75
adopting_body пар: 219793 уникальных: 3252
